In [ ]:
!pip install kaggle
!mkdir -p ~/.kaggle
!cp /content/kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json
print("Kaggle API ready")


cp: cannot stat '/content/kaggle.json': No such file or directory
chmod: cannot access '/root/.kaggle/kaggle.json': No such file or directory
Kaggle API ready


In [ ]:
!ls ~/.kaggle

In [ ]:
!kaggle datasets download -d beksod/ai-generated-images-vs-real-images -p /content

Traceback (most recent call last):
  File "/usr/local/bin/kaggle", line 10, in <module>
    sys.exit(main())
             ^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/kaggle/cli.py", line 68, in main
    out = args.func(**command_args)
          ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/kaggle/api/kaggle_api_extended.py", line 1741, in dataset_download_cli
    with self.build_kaggle_client() as kaggle:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/kaggle/api/kaggle_api_extended.py", line 688, in build_kaggle_client
    username=self.config_values['username'],
             ~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^
KeyError: 'username'


In [ ]:
!pip install kagglehub


In [ ]:
import kagglehub

dataset_path = kagglehub.dataset_download(
    "cashbowman/ai-generated-images-vs-real-images"
)

print(dataset_path)


100%|██████████| 476M/476M [00:05<00:00, 90.9MB/s]

Extracting files...


/root/.cache/kagglehub/datasets/cashbowman/ai-generated-images-vs-real-images/versions/1


In [ ]:
import os  # used to work with files and folders

# Path where KaggleHub downloaded the dataset
DATASET_PATH = "/root/.cache/kagglehub/datasets/cashbowman/ai-generated-images-vs-real-images/versions/1"

# List all folders/files inside the dataset directory
print(os.listdir(DATASET_PATH))

['AiArtData', 'RealArt']


In [ ]:
# Path where the Kaggle dataset is stored
DATASET_PATH = "/root/.cache/kagglehub/datasets/cashbowman/ai-generated-images-vs-real-images/versions/1"

# Path to the folder containing REAL images
REAL_PATH = DATASET_PATH + "/RealArt"

# Path to the folder containing AI-GENERATED images
AI_PATH = DATASET_PATH + "/AiArtData"

In [ ]:
# Path to the REAL images folder
# (Images are inside a nested folder: RealArt/RealArt)
REAL_PATH = DATASET_PATH + "/RealArt/RealArt"

# Path to the AI-GENERATED images folder
# (Images are inside a nested folder: AiArtData/AiArtData)
AI_PATH = DATASET_PATH + "/AiArtData/AiArtData"

In [ ]:
import os           # for accessing folders and files
import cv2          # for image reading and processing
import numpy as np  # for numerical operations

IMG_SIZE = 224      # standard size for all images

# Function to load images from a folder
def load_images(folder, label):
    data = []  # list to store images and labels

    # Loop through all files in the given folder
    for file in os.listdir(folder):

        # Check if the file is an image
        if file.lower().endswith(('.jpg', '.png', '.jpeg')):

            # Create full path of the image
            path = os.path.join(folder, file)

            # Read the image
            img = cv2.imread(path)

            # Skip if image cannot be read
            if img is None:
                continue

            # Resize image to fixed size
            img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))

            # Normalize pixel values (0–1)
            img = img / 255.0

            # Store image with its label
            data.append([img, label])

    return data  # return processed images

# Load REAL images (label = 0)
data = []
data += load_images(REAL_PATH, 0)

# Load AI-generated images (label = 1)
data += load_images(AI_PATH, 1)

# Shuffle the dataset to mix real and AI images
np.random.shuffle(data)

# Separate features (X) and labels (y)
X = np.array([i[0] for i in data])  # images
y = np.array([i[1] for i in data])  # labels

# Print dataset shape
print(X.shape, y.shape)

(970, 224, 224, 3) (970,)


In [ ]:
from sklearn.model_selection import train_test_split
# used to split dataset into training and testing parts

# Split images and labels into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y,              # full dataset (images and labels)
    test_size=0.2,     # 20% data for testing, 80% for training
    random_state=42    # fixed value for reproducible results
)

# Print shapes to confirm the split
print(X_train.shape, X_test.shape)

(776, 224, 224, 3) (194, 224, 224, 3)


In [ ]:
import tensorflow as tf  # TensorFlow for building and training neural networks

# Create a Sequential CNN model
model = tf.keras.models.Sequential([

    # Input layer defining image size and color channels (RGB)
    tf.keras.layers.Input(shape=(224, 224, 3)),

    # First convolution layer to extract basic features (edges, textures)
    tf.keras.layers.Conv2D(32, (3,3), activation='relu'),

    # Reduces image size and computation
    tf.keras.layers.MaxPooling2D(2,2),

    # Second convolution layer to extract deeper features
    tf.keras.layers.Conv2D(64, (3,3), activation='relu'),

    # Further downsampling
    tf.keras.layers.MaxPooling2D(2,2),

    # Converts 2D feature maps into a 1D vector
    tf.keras.layers.Flatten(),

    # Fully connected layer for learning complex patterns
    tf.keras.layers.Dense(128, activation='relu'),

    # Output layer:
    # 1 neuron + sigmoid → binary classification (AI or Real)
    tf.keras.layers.Dense(1, activation='sigmoid')
])

# Compile the model
model.compile(
    optimizer='adam',                 # Adam optimizer for faster convergence
    loss='binary_crossentropy',       # Loss function for binary classification
    metrics=['accuracy']              # Metric to evaluate model performance
)

In [ ]:
# Train the model using training data
model.fit(
    X_train,      # input images for training
    y_train,      # labels (0 = Real, 1 = AI)

    epochs=5,     # number of times the model sees the full dataset

    validation_data=(X_test, y_test)
    # validation data to check accuracy on unseen images
)


Epoch 1/5
25/25 ━━━━━━━━━━━━━━━━━━━━ 11s 215ms/step - accuracy: 0.5125 - loss: 2.6561 - val_accuracy: 0.5464 - val_loss: 0.6793
Epoch 2/5
25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 51ms/step - accuracy: 0.6016 - loss: 0.6470 - val_accuracy: 0.6134 - val_loss: 0.6686
Epoch 3/5
25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 50ms/step - accuracy: 0.7161 - loss: 0.5612 - val_accuracy: 0.6443 - val_loss: 0.6388
Epoch 4/5
25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 50ms/step - accuracy: 0.8554 - loss: 0.4022 - val_accuracy: 0.6649 - val_loss: 0.6651
Epoch 5/5
25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 50ms/step - accuracy: 0.9323 - loss: 0.2341 - val_accuracy: 0.6804 - val_loss: 0.6881


In [ ]:
model.save("ai_image_detector.keras")

In [ ]:
import tensorflow as tf

model = tf.keras.models.load_model("ai_image_detector.keras")
print("Model loaded")

Model loaded


/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:802: UserWarning: Skipping variable loading for optimizer 'rmsprop', because it has 10 variables whereas the saved optimizer has 18 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [ ]:
import cv2, numpy as np

def predict_image(image_path):
    # Read image
    img = cv2.imread(image_path)

    # Resize to model input size
    img = cv2.resize(img, (224, 224))

    # Normalize pixel values
    img = img / 255.0

    # Add batch dimension
    img = np.expand_dims(img, axis=0)

    # Get prediction
    prediction = model.predict(img)[0][0]

    # Decision + confidence
    if prediction >= 0.5:
        return "AI-Generated Image", round(prediction * 100, 2)
    else:
        return "Real Image", round((1 - prediction) * 100, 2)

In [ ]:
# Test with a real image
test_image = REAL_PATH + "/" + os.listdir(REAL_PATH)[0]
predict_image(test_image)


1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step


('Real Image', np.float32(88.95))

In [ ]:
import os

os.makedirs("/content/drive/MyDrive/AI_Image_Project", exist_ok=True)
print("Folder created")


Folder created


In [ ]:
model.save("/content/drive/MyDrive/AI_Image_Project/ai_image_detector.keras")
print("Model saved to Drive")


Model saved to Drive
